In [10]:
import os, sys, pickle, importlib, gc
import pandas as pd
import numpy as np

sys.path.insert(0, os.path.abspath('../../'))
import src.logit_graph.simulation
sys.modules['src.simulation'] = src.logit_graph.simulation

In [18]:
DATASETS = {
    'Facebook':    'runs/fitted_graphs_comparison_facebook5',
    # 'Twitter':     'runs/fitted_graphs_comparison_twitter2',
    # 'Google Plus': 'runs/fitted_graphs_comparison_gplus1',
    # 'Reddit':      'runs/fitted_graphs_comparison_reddit3',
    'Twitch':      'runs/fitted_graphs_comparison_twitch',
}

In [19]:
def load_summary_dfs(folder):
    """Load all comparator summary DataFrames from a runs folder.

    Each pkl file contains a growing list of GraphModelComparator objects.
    The last element in each file is the comparator for the graph named in
    the filename, so we load every file and take its final element.
    After collecting all summary DataFrames we de-duplicate by graph_filename
    (keeping the last occurrence) to guard against overlapping saves.
    """
    pkl_files = sorted(
        [f for f in os.listdir(folder)
         if f.endswith('.pkl') and 'comparators' in f.lower()]
    )
    if not pkl_files:
        print(f'  No comparator pkl files found in {folder}')
        return []

    dfs = []
    for fname in pkl_files:
        path = os.path.join(folder, fname)
        with open(path, 'rb') as fh:
            data = pickle.load(fh)

        if isinstance(data, list) and len(data) > 0:
            comp = data[-1]
        else:
            comp = data

        if hasattr(comp, 'summary_df') and comp.summary_df is not None:
            dfs.append(comp.summary_df)

        del data, comp
        gc.collect()

    seen = {}
    for df in dfs:
        gname = df['graph_filename'].iloc[0]
        seen[gname] = df
    return list(seen.values())

In [21]:
from tqdm import tqdm

# Initialize an empty list to store results for each graph
rows = []

# Iterate through each dataset and its corresponding folder
for dataset_name, folder in DATASETS.items():
    print(f'\n{"="*60}')
    print(f'Processing dataset: {dataset_name}')
    print(f'Looking in folder: {folder}')
    print(f'{"="*60}')
    
    # Check if the folder exists before attempting to load data
    if not os.path.isdir(folder):
        print(f'  ⚠ WARNING: Folder not found — skipping this dataset')
        continue

    # Load all summary DataFrames from the comparator pickle files
    print(f'  Loading summary DataFrames from comparator files...')
    summary_dfs = load_summary_dfs(folder)
    print(f'  ✓ Successfully loaded {len(summary_dfs)} graph(s)')

    # Process each graph's summary DataFrame
    for idx, sdf in enumerate(tqdm(summary_dfs, desc=f'Processing {dataset_name} graphs'), start=1):
        # Extract the graph identifier from the filename
        graph_filename = sdf['graph_filename'].iloc[0]
        graph_id = graph_filename.replace('.edges', '')
        
        # Filter for the Logistic Graph (LG) model row
        lg_row = sdf[sdf['model'] == 'LG']
        
        # Extract the GIC value if the LG model exists, otherwise use NaN
        if not lg_row.empty:
            gic = lg_row['gic_value'].iloc[0]
            print(f'    Graph {idx}/{len(summary_dfs)}: {graph_id} → GIC = {gic:.4f}')
        else:
            gic = np.nan
            print(f'    Graph {idx}/{len(summary_dfs)}: {graph_id} → GIC = N/A (LG model not found)')
        
        # Append the results to our collection
        rows.append({
            'dataset': dataset_name,
            'graph_id': graph_id,
            'gic': gic
        })

# Convert the collected results into a pandas DataFrame
print(f'\n{"="*60}')
print(f'Creating consolidated results DataFrame...')
results_df = pd.DataFrame(rows)
print(f'✓ Successfully created DataFrame with {len(results_df)} total graph(s)')
print(f'{"="*60}\n')

# Display the results DataFrame
results_df


Processing dataset: Facebook
Looking in folder: runs/fitted_graphs_comparison_facebook5
  Loading summary DataFrames from comparator files...
  ✓ Successfully loaded 10 graph(s)


Processing Facebook graphs: 100%|██████████| 10/10 [00:00<00:00, 5319.35it/s]

    Graph 1/10: 0 → GIC = 1.0938
    Graph 2/10: 107 → GIC = 0.9514
    Graph 3/10: 1684 → GIC = 0.9042
    Graph 4/10: 1912 → GIC = 0.6606
    Graph 5/10: 3437 → GIC = 1.7604
    Graph 6/10: 348 → GIC = 1.8530
    Graph 7/10: 3980 → GIC = 0.5967
    Graph 8/10: 414 → GIC = 1.0400
    Graph 9/10: 686 → GIC = 0.7852
    Graph 10/10: 698 → GIC = 0.5263

Processing dataset: Twitch
Looking in folder: runs/fitted_graphs_comparison_twitch
  Loading summary DataFrames from comparator files...


  ✓ Successfully loaded 6 graph(s)


Processing Twitch graphs: 100%|██████████| 6/6 [00:00<00:00, 3932.16it/s]

    Graph 1/6: DE_graph → GIC = 0.1843
    Graph 2/6: ENGB_graph → GIC = 0.1852
    Graph 3/6: ES_graph → GIC = 0.2486
    Graph 4/6: FR_graph → GIC = 0.2157
    Graph 5/6: PTBR_graph → GIC = 0.3282
    Graph 6/6: RU_graph → GIC = 0.1896

Creating consolidated results DataFrame...
✓ Successfully created DataFrame with 16 total graph(s)



,dataset,graph_id,gic
0,Facebook,0,1.093779
1,Facebook,107,0.951384
2,Facebook,1684,0.904223
3,Facebook,1912,0.660607
4,Facebook,3437,1.760406
5,Facebook,348,1.853001
6,Facebook,3980,0.596708
7,Facebook,414,1.039963
8,Facebook,686,0.785177
9,Facebook,698,0.526334


In [ ]:
results_df.to_csv('consolidated_gic_results.csv', index=False)
print('Saved to consolidated_gic_results.csv')